# Fine-tune Qwen2.5-3B-Instruct bang DeLoRA tren VlogQA

| Tham so | LoRA | **DeLoRA** |
|---|---|---|
| PEFT config | `LoraConfig` | **`DeloraConfig`** |
| Boundary param | N/A | **`delora_lambda=15`** |
| Dropout | `lora_dropout` | **`module_dropout=0.05`** |
| Learning Rate | `2e-4` | **`5e-3`** (cao hon 25x) |
| Quantization | QLoRA OK | **Phai BF16 full precision** |

**Hardware:** RTX 3090 (24GB VRAM)  
**Dataset:** VlogQA (Extractive QA tieng Viet)  
**Tracking:** Weights & Biases (W&B)

## Buoc 0: Cai dat thu vien

In [ ]:
!pip uninstall torch torchvision torchaudio xformers unsloth -y 2>/dev/null || true

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
# PEFT >= 0.19.0 co DeloraConfig chinh thuc
!pip install -q \
    "peft>=0.19.0" \
    "transformers>=4.45.0" \
    "trl>=0.9.0" \
    "accelerate>=0.34.0" \
    "datasets>=2.21.0" \
    "wandb"

In [ ]:
import peft, torch, trl, transformers
print(f"PEFT         : {peft.__version__}")
print(f"TRL          : {trl.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
from peft import DeloraConfig
print("DeloraConfig : OK")

## Buoc 0.5: Cau hinh toan cuc (CFG) - Single Source of Truth

In [ ]:
# ==========================================
# HARDWARE & PERFORMANCE MONITOR (PACLIC 2026)
# ==========================================
!pip install -q pynvml
import threading, time, pynvml
import torch

class HardwareMonitor:
    def __init__(self):
        try:
            pynvml.nvmlInit()
            self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            self.has_nvml = True
        except Exception as e:
            print(f"NVML Init failed: {e}")
            self.has_nvml = False
        self.is_running = False
        self.power_readings = []
    
    def _monitor(self):
        while self.is_running:
            if self.has_nvml:
                power = pynvml.nvmlDeviceGetPowerUsage(self.handle) / 1000.0
                self.power_readings.append(power)
            time.sleep(0.5)
            
    def start(self):
        self.is_running = True
        self.power_readings = []
        self.thread = threading.Thread(target=self._monitor)
        self.thread.start()
        
    def stop(self):
        self.is_running = False
        if hasattr(self, 'thread'):
            self.thread.join()
        avg_power = sum(self.power_readings)/len(self.power_readings) if self.power_readings else 0
        peak_vram = torch.cuda.max_memory_allocated() / 1e9
        return avg_power, peak_vram

hw_monitor = HardwareMonitor()
print("Hardware Monitor Initialized!")


In [ ]:
# ============================================================
# CFG - Toan bo hyperparameters o mot cho duy nhat
# Moi thay doi chi can sua o day, tu dong dong bo voi W&B
# ============================================================
CFG = {
    # --- Model ---
    "model_name"       : "Qwen/Qwen2.5-3B-Instruct",

    # --- DeLoRA ---
    "peft_method"      : "delora",
    "r"                : 16,
    "delora_lambda"    : 15,
    "module_dropout"   : 0.05,
    "target_modules"   : ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],

    # --- Training ---
    "learning_rate"              : 5e-3,
    "lr_scheduler_type"          : "cosine",
    "warmup_ratio"               : 0.03,
    "num_train_epochs"           : 5,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size" : 4,
    "gradient_accumulation_steps": 4,
    "weight_decay"               : 0.01,
    "optim"                      : "adamw_torch",
    "seed"                       : 3407,

    # --- Sequence ---
    "max_seq_length"             : 8192,
    "max_context_tokens"         : 7500,

    # --- Early Stopping ---
    "early_stopping_patience"    : 5,
    "eval_steps"                 : 100,     # Danh gia moi 100 steps

    # --- Paths ---
    "train_path"   : "../../../train.json",
    "dev_path"     : "../../../dev.json",
    "test_path"    : "../../../test.json",
    "save_path"    : "qwen2.5-3b-instruct-delora-vlogqa",
    "checkpoint_dir": "outputs_delora_vlogqa",
    "result_file"  : "delora_vlogqa_test_results.json",

    # --- W&B ---
    "wandb_project"  : "PACLIC_2026-VlogQA",
    "wandb_run_name" : "qwen2.5-3b-delora-vlogqa",
    "wandb_tags"     : ["delora", "qwen2.5-3b", "vlogqa", "paclic2026"],
}

print("CFG initialized:")
for k, v in CFG.items():
    print(f"  {k:30s}: {v}")

## Buoc 0.6: Khoi tao Weights & Biases

In [ ]:
# ==========================================
# CAI DAT VA KHOI TAO WEIGHTS & BIASES (W&B)
# ==========================================
import wandb
import os

wandb.login()  # Se hoi API key neu chua login

# Truyen thang CFG vao wandb.init -> luon dong bo, khong bao gio lech
wandb_run = wandb.init(
    project = CFG["wandb_project"],
    name    = CFG["wandb_run_name"],
    config  = CFG,   # <-- toan bo CFG, single source of truth
    tags    = CFG["wandb_tags"],
)

print(f"[W&B] Run initialized: {wandb_run.url}")
print(f"  Project : {CFG['wandb_project']}")
print(f"  Run name: {CFG['wandb_run_name']}")

## 1. Load Model va Tokenizer (Full BF16 - Khong QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Dang tai tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG["model_name"],
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Tu dong detect FlashAttention2
try:
    import importlib
    importlib.import_module("flash_attn")
    _attn_impl = "flash_attention_2"
    print("FlashAttention2 kha dung - dung flash_attention_2")
except ImportError:
    _attn_impl = "sdpa"  # Scaled Dot-Product Attention (PyTorch 2.x built-in)
    print("FlashAttention2 chua cai - fallback sang sdpa")

print(f"Dang tai model (BF16, khong quantize)...")
model = AutoModelForCausalLM.from_pretrained(
    CFG["model_name"],
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation=_attn_impl,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"\nModel da load thanh cong!")
print(f"  Dtype  : {next(model.parameters()).dtype}")
print(f"  Params : {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"  Attn   : {_attn_impl}")
print(f"  VRAM   : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 2. Cau hinh DeLoRA Adapter

In [ ]:
from peft import get_peft_model, DeloraConfig

delora_config = DeloraConfig(
    r              = CFG["r"],
    delora_lambda  = CFG["delora_lambda"],
    target_modules = CFG["target_modules"],
    module_dropout = CFG["module_dropout"],
    bias           = "none",
    task_type      = "CAUSAL_LM",
    init_weights   = True,
)

model = get_peft_model(model, delora_config)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

print("Cau hinh DeLoRA thanh cong!")
model.print_trainable_parameters()

## 3. Chuan bi Dataset VlogQA

In [ ]:
import json
from datasets import Dataset

def load_vlogqa(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                answer = qa["answers"][0]["text"] if qa["answers"] else ""
                if answer:
                    samples.append({"context": context, "question": question,
                                    "answer": answer, "id": qa.get("id", "")})
    return samples


def truncate_context_around_answer(context, answer, tokenizer,
                                   max_ctx_tokens=CFG["max_context_tokens"]):
    answer_start = context.find(answer)
    if answer_start == -1:
        ctx_ids = tokenizer.encode(context, add_special_tokens=False)
        return context if len(ctx_ids) <= max_ctx_tokens else \
               tokenizer.decode(ctx_ids[:max_ctx_tokens], skip_special_tokens=True)
    before_ids = tokenizer.encode(context[:answer_start], add_special_tokens=False)
    answer_ids = tokenizer.encode(answer, add_special_tokens=False)
    after_ids  = tokenizer.encode(context[answer_start + len(answer):], add_special_tokens=False)
    if len(before_ids) + len(answer_ids) + len(after_ids) <= max_ctx_tokens:
        return context
    budget = max_ctx_tokens - len(answer_ids)
    half   = budget // 2
    return tokenizer.decode(before_ids[-half:] + answer_ids + after_ids[:budget - half],
                            skip_special_tokens=True)


SYSTEM_PROMPT = (
    "Ban la he thong trich xuat cau tra loi tu van ban tieng Viet. "
    "QUY TAC BAT BUOC:\n"
    "1) Chi tra ve DUNG cum tu xuat hien nguyen van trong doan van.\n"
    "2) Khong viet cau hoan chinh, khong giai thich, khong them tien to nao.\n"
    "3) Cau tra loi phai CO MAT trong doan van."
)
USER_PROMPT_TEMPLATE = (
    "Trich xuat cau tra loi tu doan van. Chi tra ve cum tu trong doan van.\n\n"
    "Doan van:\n{context}\n\nCau hoi: {question}\n\nCau tra loi (span-only):"
)


def format_prompt_train(context, question, answer, tokenizer):
    context_cropped = truncate_context_around_answer(context, answer, tokenizer)
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": USER_PROMPT_TEMPLATE.format(
            context=context_cropped, question=question)},
        {"role": "assistant", "content": answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def format_prompt_inference(context, question, tokenizer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT_TEMPLATE.format(
            context=context[:CFG["max_context_tokens"]], question=question)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("Dinh nghia ham xong!")

In [ ]:
from datasets import Dataset

train_samples = load_vlogqa(CFG["train_path"])
print(f"Train: {len(train_samples)} mau")

train_texts = [format_prompt_train(s["context"], s["question"], s["answer"], tokenizer)
               for s in train_samples]
dataset = Dataset.from_dict({"text": train_texts})

dev_samples = load_vlogqa(CFG["dev_path"])
print(f"Dev  : {len(dev_samples)} mau")

dev_texts = [format_prompt_train(s["context"], s["question"], s["answer"], tokenizer)
             for s in dev_samples]
eval_dataset = Dataset.from_dict({"text": dev_texts})

print(f"\nVi du prompt (150 ky tu dau): {train_texts[0][:150]}...")

## 4. Cau hinh va Bat dau Huan luyen

> **Fix `SFTConfig.__init__() got an unexpected keyword argument 'max_seq_length'`**  
> `max_seq_length`, `dataset_text_field`, `packing`, `dataset_num_proc`  
> phai truyen vao **`SFTTrainer`** truc tiep, KHONG phai `TrainingArguments`.

In [ ]:
# ============================================================
# TRL 0.24.0 API:
#   - Dung SFTConfig (ke thua TrainingArguments), KHONG dung TrainingArguments truc tiep
#   - max_seq_length da bi xoa khoi SFTConfig -> set qua tokenizer.model_max_length
#   - packing=True van con trong SFTConfig
#   - dataset_text_field con trong SFTConfig (neu co), fallback sang formatting_func
#   - SFTTrainer chi nhan: model, args, datasets, processing_class, formatting_func, callbacks
# ============================================================
import inspect
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
import sys

# Set max sequence length qua tokenizer (cach dung trong TRL >= 0.12)
tokenizer.model_max_length = CFG["max_seq_length"]
print(f"tokenizer.model_max_length = {tokenizer.model_max_length}")

# Kiem tra nhung params nao SFTConfig chap nhan
sft_cfg_params = set(inspect.signature(SFTConfig.__init__).parameters.keys())
print(f"SFTConfig has packing          : {'packing' in sft_cfg_params}")
print(f"SFTConfig has dataset_text_field: {'dataset_text_field' in sft_cfg_params}")
print(f"SFTConfig has max_seq_length   : {'max_seq_length' in sft_cfg_params}")

# ============================================================
# SFTConfig: TrainingArguments + SFT-specific params
# ============================================================
sft_cfg_kwargs = dict(
    # --- Batch ---
    per_device_train_batch_size  = CFG["per_device_train_batch_size"],
    per_device_eval_batch_size   = CFG["per_device_eval_batch_size"],
    gradient_accumulation_steps  = CFG["gradient_accumulation_steps"],

    # --- DeLoRA LR ---
    learning_rate                = CFG["learning_rate"],
    lr_scheduler_type            = CFG["lr_scheduler_type"],
    warmup_ratio                 = CFG["warmup_ratio"],

    # --- Epochs ---
    num_train_epochs             = CFG["num_train_epochs"],

    # --- Precision ---
    bf16                         = True,
    fp16                         = False,

    # --- Optimizer ---
    optim                        = CFG["optim"],
    weight_decay                 = CFG["weight_decay"],

    # --- Gradient Checkpointing ---
    gradient_checkpointing       = True,
    gradient_checkpointing_kwargs= {"use_reentrant": False},

    # --- W&B Logging ---
    logging_steps                = 20,
    report_to                    = "wandb",
    run_name                     = CFG["wandb_run_name"],

    # --- Evaluation & Checkpoint ---
    eval_strategy                = "steps",
    save_strategy                = "steps",
    eval_steps                   = CFG["eval_steps"],
    save_steps                   = CFG["eval_steps"],
    load_best_model_at_end       = True,
    metric_for_best_model        = "eval_loss",
    greater_is_better            = False,
    save_total_limit             = 2,

    # --- Output ---
    output_dir                   = CFG["checkpoint_dir"],
    seed                         = CFG["seed"],
    dataloader_num_workers       = 0,
)

# Them packing neu SFTConfig ho tro (TRL 0.24 co)
if "packing" in sft_cfg_params:
    sft_cfg_kwargs["packing"] = True

# Them dataset_text_field neu SFTConfig ho tro
if "dataset_text_field" in sft_cfg_params:
    sft_cfg_kwargs["dataset_text_field"] = "text"

training_args = SFTConfig(**sft_cfg_kwargs)

# ============================================================
# formatting_func: fallback neu dataset_text_field khong co
# ============================================================
sft_trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())

trainer_kwargs = dict(
    model         = model,
    args          = training_args,
    train_dataset = dataset,
    eval_dataset  = eval_dataset,
    processing_class = tokenizer,
    callbacks     = [EarlyStoppingCallback(
        early_stopping_patience=CFG["early_stopping_patience"])],
)

# Neu dataset_text_field khong co trong SFTConfig -> dung formatting_func
if "dataset_text_field" not in sft_cfg_params and "formatting_func" in sft_trainer_params:
    trainer_kwargs["formatting_func"] = lambda x: x["text"]
    print("[Info] Dung formatting_func (TRL 0.24+ khong co dataset_text_field trong SFTConfig)")

trainer = SFTTrainer(**trainer_kwargs)

# Fix sys.modules de tranh loi pickle
real_config_cls  = type(trainer.args)
real_trainer_cls = type(trainer)
sys.modules['trl.trainer.sft_config']  = sys.modules[real_config_cls.__module__]
sys.modules['trl.trainer.sft_trainer'] = sys.modules[real_trainer_cls.__module__]
sys.modules[real_config_cls.__module__].SFTConfig   = real_config_cls
sys.modules[real_trainer_cls.__module__].SFTTrainer = real_trainer_cls

eff_batch = CFG["per_device_train_batch_size"] * CFG["gradient_accumulation_steps"]
print(f"\nTrainer ready! (TRL {__import__('trl').__version__})")
print(f"  Effective batch  : {eff_batch}")
print(f"  Learning rate    : {CFG['learning_rate']}")
print(f"  Max seq length   : {tokenizer.model_max_length} (set via tokenizer)")
print(f"  Packing          : {getattr(training_args, 'packing', 'N/A')}")
print(f"  W&B run          : {wandb_run.url}")


In [ ]:
import torch

print(f"VRAM truoc train: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"\nBat dau fine-tuning DeLoRA (patience={CFG['early_stopping_patience']}, max {CFG['num_train_epochs']} epochs)...\n")

hw_monitor.start()
trainer_stats = trainer.train()
avg_power, peak_vram = hw_monitor.stop()

train_time = trainer_stats.metrics.get('train_runtime', 0)

print(f"\n{'='*50}")
print(f"Hoan tat Training!")
print(f"  Epochs da chay   : {trainer_stats.metrics.get('epoch', 'N/A'):.2f}")
print(f"  Train Loss cuoi  : {trainer_stats.metrics.get('train_loss', 'N/A'):.4f}")
print(f"  Training Time    : {train_time/60:.2f} phut")
print(f"  Peak GPU VRAM    : {peak_vram:.2f} GB")
print(f"  Avg GPU Power    : {avg_power:.2f} W")
print(f"{'='*50}")

# Log vao wandb
import wandb
if wandb.run is not None:
    wandb.log({
        "hardware/train_peak_vram_gb": peak_vram,
        "hardware/train_avg_power_w": avg_power,
        "hardware/train_runtime_min": train_time/60
    })
    # Lưu ID của W&B run lại để nếu Restart Kernel vẫn resume được ở test cell
    with open("wandb_run_id.txt", "w") as f:
        f.write(wandb.run.id)
    print(f"[W&B] Run ID saved: {wandb.run.id}. Run is STILL ACTIVE for testing.")


## 5. Luu Model

In [ ]:
model.save_pretrained(CFG["save_path"])
tokenizer.save_pretrained(CFG["save_path"])

print(f"Da luu mo hinh DeLoRA tai: {CFG['save_path']}")
import os
for f in sorted(os.listdir(CFG["save_path"])):
    size = os.path.getsize(os.path.join(CFG["save_path"], f))
    print(f"  {f:40s} {size/1e6:.1f} MB")

## 6. Kiem tra Inference nhanh

In [ ]:
import torch

model.eval()
model.config.use_cache = True

sample = train_samples[0]
prompt = format_prompt_inference(sample["context"], sample["question"], tokenizer)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=64, do_sample=False,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("\n--- KET QUA INFERENCE ---")
print(f"Cau hoi      : {sample['question']}")
print(f"Dap an dung  : {sample['answer']}")
print(f"Model tra loi: {response}")
print(f"Exact Match  : {response.strip() == sample['answer'].strip()}")

## 7. Danh gia day du tren tap Test (EM & F1)

In [ ]:
import re, json, string
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader

def normalize_answer(s):
    s = s.lower()
    s = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', s)
    return ' '.join(s.split())

def compute_exact_match(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))

def compute_f1(pred, gold):
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    common = set(pred_tokens) & set(gold_tokens)
    if not common: return 0.0
    precision = len(common) / len(pred_tokens)
    recall    = len(common) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

test_samples = load_vlogqa(CFG["test_path"])
print(f"Test set: {len(test_samples)} mau")

model.eval()
model.config.use_cache = True

# Set padding to left for batched inference
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

results, em_scores, f1_scores = [], [], []
BATCH_SIZE = 8  # Chay 8 sample cung luc cho nhanh

for i in tqdm(range(0, len(test_samples), BATCH_SIZE), desc="Evaluating (Batched)"):
    batch_samples = test_samples[i : i + BATCH_SIZE]
    prompts = [
        format_prompt_inference(s["context"], s["question"], tokenizer)
        for s in batch_samples
    ]
    
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, 
        truncation=True, max_length=CFG["max_seq_length"]
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=64, do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
        
    # Tinh chieu dai dau vao cua tung sample de extract prediction chinh xac
    for j, s in enumerate(batch_samples):
        input_len = inputs["input_ids"][j].shape[-1]
        generated_ids = outputs[j][input_len:]
        prediction = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        
        em = compute_exact_match(prediction, s["answer"])
        f1 = compute_f1(prediction, s["answer"])
        em_scores.append(em)
        f1_scores.append(f1)
        results.append({"id": s["id"], "question": s["question"],
                        "gold": s["answer"], "prediction": prediction, "em": em, "f1": f1})

# Restore tokenizer padding side to right for future training if needed
tokenizer.padding_side = "right"

avg_em = sum(em_scores) / len(em_scores) * 100
avg_f1 = sum(f1_scores) / len(f1_scores) * 100

print(f"\n{'='*50}")
print(f"  Exact Match (EM): {avg_em:.2f}%")
print(f"  F1 Score        : {avg_f1:.2f}%")
print(f"{'='*50}")

# Log ket qua test len W&B
import wandb
if wandb.run is not None:
    wandb.log({"test/exact_match": avg_em, "test/f1": avg_f1,
               "test/num_samples": len(test_samples)})

with open(CFG["result_file"], "w", encoding="utf-8") as f:
    json.dump({
        "model": CFG["model_name"], "adapter": CFG["save_path"],
        "peft_method": "DeLoRA",
        "delora_config": {k: CFG[k] for k in ["r","delora_lambda","module_dropout","learning_rate"]},
        "exact_match": avg_em, "f1": avg_f1,
        "num_test_samples": len(test_samples), "predictions": results,
    }, f, ensure_ascii=False, indent=2)

print(f"\nDa luu ket qua tai: {CFG['result_file']}")


## 8. (Optional) Load lai adapter de inference sau nay

In [ ]:
# ==========================================
# CELL NAY CO THE CHAY DOC LAP SAU KHI RESTART KERNEL
# (Chi can ban da chay cell CFG o Buoc 0.5 truoc do)
# ==========================================
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
import importlib

# Tu dong detect FlashAttention2 de tang toc do inference
try:
    importlib.import_module("flash_attn")
    _attn_impl = "flash_attention_2"
    print("FlashAttention2 kha dung - bat flash_attention_2 de tang toc!")
except ImportError:
    _attn_impl = "sdpa"
    print("FlashAttention2 chua cai - dung sdpa")

print("Dang load base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    CFG["model_name"], torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True,
    attn_implementation=_attn_impl,
)
print("Dang load va merge DeLoRA adapter de tang toc do generate...")
model_loaded = PeftModel.from_pretrained(base_model, CFG["save_path"])
# Merge adapter vao base model luon -> Inference cuc nhanh, khong bi overhead cua PEFT
model_loaded = model_loaded.merge_and_unload()

tokenizer_loaded = AutoTokenizer.from_pretrained(CFG["save_path"])

print(f"Load va Merge thanh cong adapter tu: {CFG['save_path']}")


## 9. Chay test thu voi model da load

In [ ]:
# ==========================================
# CELL NAY HOAN TOAN DOC LAP (CHUA FULL DATA & METRICS)
# ==========================================
import json, re, string, time
import unicodedata
from collections import Counter
import torch
from tqdm.auto import tqdm

# --- 1. Dinh nghia lai cac ham can thiet de chay doc lap ---
def load_vlogqa(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                answer = qa["answers"][0]["text"] if qa["answers"] else ""
                if answer:
                    samples.append({"context": context, "question": question,
                                    "answer": answer, "id": qa.get("id", "")})
    return samples

SYSTEM_PROMPT = (
    "Ban la he thong trich xuat cau tra loi tu van ban tieng Viet. "
    "QUY TAC BAT BUOC:\n"
    "1) Chi tra ve DUNG cum tu xuat hien nguyen van trong doan van.\n"
    "2) Khong viet cau hoan chinh, khong giai thich, khong them tien to nao.\n"
    "3) Cau tra loi phai CO MAT trong doan van."
)
USER_PROMPT_TEMPLATE = (
    "Trich xuat cau tra loi tu doan van. Chi tra ve cum tu trong doan van.\n\n"
    "Doan van:\n{context}\n\nCau hoi: {question}\n\nCau tra loi (span-only):"
)

PREFIX_RE = re.compile(
    r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn|trả lời|span-only)\s*[:\-]?\s*",
    re.IGNORECASE,
)

def format_prompt_inference(context, question, tokenizer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT_TEMPLATE.format(
            context=context[:CFG["max_context_tokens"]], question=question)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text or "")
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())

def compute_exact_match(prediction: str, ground_truth: str) -> int:
    return int(normalize_text(prediction) == normalize_text(ground_truth))

def compute_f1_token(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_text(prediction).split()
    true_tokens = normalize_text(ground_truth).split()
    if len(pred_tokens) == 0 and len(true_tokens) == 0: return 1.0
    if len(pred_tokens) == 0 or len(true_tokens) == 0: return 0.0
    common = Counter(pred_tokens) & Counter(true_tokens)
    num_common = sum(common.values())
    if num_common == 0: return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(true_tokens)
    return (2 * precision * recall) / (precision + recall)

def clean_prediction(raw: str) -> str:
    pred = raw.strip().split("\n")[0].strip().strip('"\'\' ')
    return PREFIX_RE.sub("", pred).strip()

def find_best_span(prediction: str, context: str) -> str:
    pred_norm = normalize_text(prediction)
    if not pred_norm: return prediction
    pred_words = pred_norm.split()
    ctx_words  = context.split()
    n = len(pred_words)
    if n == 0 or len(ctx_words) == 0: return prediction
    best_f1   = compute_f1_token(prediction, context)
    best_span = prediction
    min_w = max(1, n // 2)
    max_w = min(2 * n + 5, len(ctx_words))
    for w in range(min_w, max_w + 1):
        for start in range(len(ctx_words) - w + 1):
            span_orig = " ".join(ctx_words[start:start + w])
            f1 = compute_f1_token(prediction, span_orig)
            if f1 > best_f1:
                best_f1   = f1
                best_span = span_orig
    return best_span if best_f1 >= 0.5 else prediction

# --- 2. Load Data ---
test_samples = load_vlogqa(CFG["test_path"])
TOTAL = len(test_samples)
print(f"Test set: {TOTAL} mau")

# --- 3. Run Inference ---
model_loaded.eval()
model_loaded.config.use_cache = True

tokenizer_loaded.padding_side = "left"
if tokenizer_loaded.pad_token is None:
    tokenizer_loaded.pad_token = tokenizer_loaded.eos_token

all_em_raw, all_f1_raw, all_em_span, all_f1_span, all_predictions = [], [], [], [], []
BATCH_SIZE = 16
SUMMARY_EVERY = 100
n_batches = (TOTAL + BATCH_SIZE - 1) // BATCH_SIZE
start_time = time.time()
hw_monitor.start()
total_gen_time = 0.0
total_gen_tokens = 0

print(f"\nBat dau danh gia tren {TOTAL} cau hoi...")
print(f"Batch size = {BATCH_SIZE} | So batch = {n_batches}\n")

pbar = tqdm(range(0, TOTAL, BATCH_SIZE), desc="Evaluating", total=n_batches)

for batch_start in pbar:
    batch = test_samples[batch_start : batch_start + BATCH_SIZE]
    prompts = [
        format_prompt_inference(s["context"], s["question"], tokenizer_loaded)
        for s in batch
    ]
    
    inputs = tokenizer_loaded(
        prompts, return_tensors="pt", padding=True, 
        truncation=True, max_length=CFG["max_seq_length"]
    ).to("cuda")
    
    total_input_len = inputs["input_ids"].shape[1]

    gen_start = time.time()
    with torch.no_grad():
        outputs = model_loaded.generate(
            **inputs, max_new_tokens=64, do_sample=False,
            repetition_penalty=1.1, use_cache=True,
            eos_token_id=tokenizer_loaded.eos_token_id,
            pad_token_id=tokenizer_loaded.pad_token_id,
        )
        
    for j, sample in enumerate(batch):
        gen_tokens = outputs[j][total_input_len:]
        total_gen_tokens += len(gen_tokens)
        raw_prediction = tokenizer_loaded.decode(gen_tokens, skip_special_tokens=True).strip()
        raw_prediction = raw_prediction.split("\n")[0].strip()
        
        cleaned_prediction = clean_prediction(raw_prediction)
        span_prediction    = find_best_span(cleaned_prediction, sample["context"])
        
        # Tinh metric voi 1 answer (string) tu data format
        truth = sample["answer"]
        em_raw  = compute_exact_match(cleaned_prediction, truth)
        f1_raw  = compute_f1_token(cleaned_prediction, truth)
        em_span = compute_exact_match(span_prediction, truth)
        f1_span = compute_f1_token(span_prediction, truth)
        
        all_em_raw.append(em_raw);   all_f1_raw.append(f1_raw)
        all_em_span.append(em_span); all_f1_span.append(f1_span)
        all_predictions.append({
            "id":               sample["id"],
            "question":         sample["question"],
            "ground_truth":     truth,
            "raw_prediction":   raw_prediction,
            "clean_prediction": cleaned_prediction,
            "span_prediction":  span_prediction,
            "em_raw": em_raw, "f1_raw": f1_raw,
            "em_span": em_span, "f1_span": f1_span,
        })
        
        idx = len(all_predictions)
        
        # Log tung cau
        tqdm.write(
            f"[{idx}/{TOTAL}] "
            f"Q: {sample['question'][:40]:<40} | "
            f"Truth: {truth[:30]:<30} | "
            f"Raw: {cleaned_prediction[:30]:<30} | "
            f"Span: {span_prediction[:30]:<30} | "
            f"EM_raw={em_raw} EM_span={em_span} "
            f"F1_raw={f1_raw:.3f} F1_span={f1_span:.3f}"
        )

    total_gen_time += (time.time() - gen_start)
    # Cap nhat pbar
    idx = len(all_predictions)
    elapsed = time.time() - start_time
    eta = (elapsed / idx) * (TOTAL - idx) if idx > 0 else 0
    cur_em = sum(all_em_span) / idx * 100
    cur_f1 = sum(all_f1_span) / idx * 100
    pbar.set_postfix({"EM_span": f"{cur_em:.1f}%", "F1_span": f"{cur_f1:.1f}%", "ETA": f"{eta/60:.1f}m"})

    # Checkpoint
    if idx % SUMMARY_EVERY < BATCH_SIZE and idx >= SUMMARY_EVERY:
        tqdm.write(f"\n{'='*60}")
        tqdm.write(f"  [Checkpoint ~{idx}/{TOTAL}] Thoi gian: {elapsed:.0f}s | ETA ~{eta/60:.1f} phut")
        tqdm.write(f"  EM  (raw):  {sum(all_em_raw)/idx*100:.2f}%  |  EM  (span): {cur_em:.2f}%")
        tqdm.write(f"  F1  (raw):  {sum(all_f1_raw)/idx*100:.2f}%  |  F1  (span): {cur_f1:.2f}%")
        tqdm.write(f"{'='*60}\n")

tokenizer_loaded.padding_side = "right"
avg_power, peak_vram = hw_monitor.stop()
total_time = time.time() - start_time

avg_em_raw = sum(all_em_raw) / TOTAL * 100
avg_f1_raw = sum(all_f1_raw) / TOTAL * 100
avg_em_span = sum(all_em_span) / TOTAL * 100
avg_f1_span = sum(all_f1_span) / TOTAL * 100

print(f"\nHoan tat! Tong thoi gian: {total_time/60:.1f} phut")
print(f"Toc do trung binh: {TOTAL/total_time:.1f} cau/giay")
tok_per_sec = total_gen_tokens / total_gen_time if total_gen_time > 0 else 0
latency_ms = (total_gen_time / TOTAL) * 1000 if TOTAL > 0 else 0
print(f"\n--- HARDWARE & PERFORMANCE METRICS ---")
print(f"  Latency per query: {latency_ms:.2f} ms")
print(f"  Tokens / Second  : {tok_per_sec:.2f} tok/s")
print(f"  Peak GPU VRAM    : {peak_vram:.2f} GB")
print(f"  Avg GPU Power    : {avg_power:.2f} W")
print(f"  PEFT Overhead    : ~0.0 GB (Da merge adapter vao base model)")
print(f"\n{'='*50}")
print(f"  EM (raw)  : {avg_em_raw:.2f}%  |  EM (span)  : {avg_em_span:.2f}%")
print(f"  F1 (raw)  : {avg_f1_raw:.2f}%  |  F1 (span)  : {avg_f1_span:.2f}%")
print(f"{'='*50}")

LOADED_RESULT_FILE = "delora_vlogqa_loaded_model_test_results.json"
with open(LOADED_RESULT_FILE, "w", encoding="utf-8") as f:
    json.dump({
        "model": CFG["model_name"], "adapter": CFG["save_path"],
        "peft_method": "DeLoRA (Loaded Model)",
        "metrics_raw":  {"em": avg_em_raw,  "f1": avg_f1_raw},
        "metrics_span": {"em": avg_em_span, "f1": avg_f1_span},
        "performance": {
            "latency_ms": latency_ms, "tok_per_sec": tok_per_sec,
            "peak_vram_gb": peak_vram, "avg_power_w": avg_power
        },
        "num_test_samples": TOTAL, "predictions": all_predictions,
    }, f, ensure_ascii=False, indent=2)

print(f"\nDa luu ket qua cua loaded model tai: {LOADED_RESULT_FILE}")

# Log test metrics vao W&B
import wandb
import os
if wandb.run is None and os.path.exists("wandb_run_id.txt"):
    with open("wandb_run_id.txt", "r") as f:
        run_id = f.read().strip()
    wandb.init(project=CFG["wandb_project"], id=run_id, resume="must")

if wandb.run is not None:
    wandb.log({
        "test/em_raw": avg_em_raw,
        "test/f1_raw": avg_f1_raw,
        "test/em_span": avg_em_span,
        "test/f1_span": avg_f1_span,
        "hardware/test_latency_ms": latency_ms,
        "hardware/test_tok_per_sec": tok_per_sec,
        "hardware/test_peak_vram_gb": peak_vram,
        "hardware/test_avg_power_w": avg_power
    })
    wandb.finish()
    print("\n[W&B] Da log test metrics va ket thuc Run!")
